<a href="https://colab.research.google.com/github/sensein/asd-ai-scoping-review/blob/update-scripts/scripts/PRISMA_pipeline_Fabio/3_local_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import pandas as pd
# Mount Google Drive
from google.colab import drive as gdrive
import re
import numpy as np
import statistics  # for calculating mean and standard deviation

In [ ]:
gdrive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define the folder containing the subfolders with CSV files
main_folder = '/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries'

In [ ]:
df_input_file = f'{main_folder}/autism_formatted__controlled_combined_noduplicates.csv'

In [ ]:
df = pd.read_csv(df_input_file)
original_data = df
df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database,text
0,"""Mommy Blogs"" and the Vaccination Exemption Na...",Social media offer an unprecedented opportunit...,"['Internet', 'attitudes', 'health knowledge', ...",10.2196/publichealth.6586,https://pubmed.ncbi.nlm.nih.gov/27876690/,"Tangherlini, Roychowdhury, Glenn, Crespi, Band...",JMIR public health and surveillance,2016.0,PubMed,"Title: ""Mommy Blogs"" and the Vaccination Exemp..."
1,"""Sequencing Matters"": Investigating Suitable A...",Social robots have been shown to be promising ...,"['action selection', 'attention skills', 'auti...",10.3389/frobt.2022.784249,https://pubmed.ncbi.nlm.nih.gov/35356059/,"Baraka, Couto, Melo, Paiva, Veloso",Frontiers in robotics and AI,2022.0,PubMed,"Title: ""Sequencing Matters"": Investigating Sui..."
2,"""Um"" and ""Uh"" Usage Patterns in Children with ...","Pragmatic language difficulties, including unu...","['Autism', 'Disfluency', 'Fillers', 'Natural l...",10.1007/s10803-022-05565-4,https://pubmed.ncbi.nlm.nih.gov/35499654/,"Lawley, Bedrick, MacFarlane, Dolata, Salem, Fo...",Journal of autism and developmental disorders,2023.0,PubMed,"Title: ""Um"" and ""Uh"" Usage Patterns in Childre..."
3,'SenseA'-Autism Early Signs and Pre-Aggressive...,This paper presents an efficient solution for ...,autism early signs; computer vision; feature e...,10.1109/AMS.2017.28,https://www.scopus.com/inward/record.uri?eid=2...,Gamaethige C.; Gunathilake U.; Jayasena D.; Ma...,AMS 2017 - Asia Modelling Symposium 2017 and 1...,2018.0,Scopus,Title: 'SenseA'-Autism Early Signs and Pre-Agg...
4,25th Annual Computational Neuroscience Meeting...,A1 Functional advantages of cell-type heteroge...,NaN,10.1186/s12868-016-0283-6,https://pubmed.ncbi.nlm.nih.gov/27534393/,NaN,BMC neuroscience,2016.0,PubMed,Title: 25th Annual Computational Neuroscience ...
...,...,...,...,...,...,...,...,...,...,...
1832,rs-fMRI Analysis Using Spatio-Temporal Sparse ...,Neuropsychiatric diseases such as Autism Spect...,CNN; Deep Learning; fMRI; Image Processing; Su...,10.1109/SIU55565.2022.9864751,https://www.scopus.com/inward/record.uri?eid=2...,Yener F.M.; Yildiz S.; Hafeez M.A.; Kayasandik...,2022 30th Signal Processing and Communications...,2022.0,Scopus,Title: rs-fMRI Analysis Using Spatio-Temporal ...
1833,rs-fMRI and machine learning for ASD diagnosis...,Autism Spectrum Disorder (ASD) diagnosis is st...,NaN,10.1038/s41598-022-09821-6,https://pubmed.ncbi.nlm.nih.gov/35411059/,"Santana, de Carvalho, Rodrigues, Bastos, de So...",Scientific reports,2022.0,PubMed,Title: rs-fMRI and machine learning for ASD di...
1834,sha-Early Intervention for children at risk of...,Autism Spectrum Disorder (ASD) is a developmen...,Autism Spectrum Disorder; CNN; Cognitive devel...,10.1109/I4Tech55392.2022.9952803,https://www.scopus.com/inward/record.uri?eid=2...,Shetty T.; Zope V.; Dandekar M.; Devnani A.; M...,2022 International Conference on Industry 4.0 ...,2022.0,Scopus,Title: sha-Early Intervention for children at ...
1835,‘Autistic Robots’ for Embodied Emulation of Be...,The goal of this work is to enable interaction...,NaN,10.1007/978-3-319-70022-9_11,https://www.scopus.com/inward/record.uri?eid=2...,Baraka K.; Melo F.S.; Veloso M.,Lecture Notes in Computer Science (including s...,2017.0,Scopus,Title: ‘Autistic Robots’ for Embodied Emulatio...


In [ ]:
"""
autism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']
ai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'ml', 'dl', 'cluster', 'clustered', 'clustering', 'classify', 'classified', 'classifying', 'classification', 'predict', 'predicted', 'predicting', 'prediction', 'recognize', 'recognized', 'recognizing', 'recognition', 'test', 'tested', 'testing', 'train', 'trained', 'training', 'model', 'modeled', 'modeling', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'nn', 'transformer', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'cv', 'natural language processing', 'nlp', 'audio signal processing', 'audio processing', 'asp', 'large language model', 'language model', 'llm', 'robot', 'robotics', 'agent', 'mining']
behavior_words = ['behavior', 'behave', 'behaved', 'behaving', 'behavioral', 'observation', 'observe', 'observing', 'observed',  'observational', 'pattern', 'response', 'respond', 'responded', 'responding', 'reaction', 'react', 'reacted', 'reacting', 'stereotype', 'stereotypical', 'stereotyped', 'repetitive', 'repeat', 'repeated', 'repeating', 'compulse', 'compulsive', 'tic', 'psychometrics', 'psychometric', 'phenotype', 'phenotypical', 'gaze', 'eye', 'fixation', 'motor', 'move', 'movement', 'moving', 'stand', 'standing', 'stood', 'crawl', 'crawled', 'crawling', 'walk', 'walking', 'walked', 'run', 'ran', 'jump', 'jumped', 'jumping', 'gait', 'locomotion', 'locomotory', 'body', 'posture', 'pose', 'gesture', 'face', 'facial', 'voice', 'speech', 'game', 'play', 'social', 'interaction', 'conversation', 'communication', 'emotion', 'emotional', 'stress', 'anxiety', 'rest', 'relax', 'sleep']
diagnosis_words = ['diagnosis', 'diagnostics', 'diagnostic', 'diagnose', 'diagnosing', 'diagnosed', 'screen', 'screened', 'screening', 'assessment', 'assess', 'assessing', 'assessed', 'detect', 'detection', 'detecting', 'detected']
"""

"\nautism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']\nai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'ml', 'dl', 'cluster', 'clustered', 'clustering', 'classify', 'classified', 'classifying', 'classification', 'predict', 'predicted', 'predicting', 'prediction', 'recognize', 'recognized', 'recognizing', 'recognition', 'test', 'tested', 'testing', 'train', 'trained', 'training', 'model', 'modeled', 'modeling', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'nn', 'transformer', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'cv', 'natural language processing', 'nlp', 'audio signal processing', 'audio processing', 'asp', 'large language model', 'language model', 'llm', 'robot', 'robotics', 'agent', 'mining']\nbeha

In [ ]:
# after brainstorming, we converged to the following set of words!
autism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']
ai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'natural language processing', 'audio signal processing', 'audio processing', 'large language model', 'language model', 'robot', 'robotics']
behavior_words = ['behavior', 'behave', 'behaved', 'behaving', 'behavioral', 'observation', 'observe', 'observing', 'observed', 'observational', 'pattern', 'response', 'respond', 'responded', 'responding', 'reaction', 'react', 'reacted', 'reacting', 'psychometrics', 'psychometric', 'phenotype', 'phenotypical', 'stereotype', 'stereotypical', 'stereotyped', 'repetitive', 'repeat', 'repeated', 'repeating', 'compulse', 'compulsive', 'tic', 'gaze', 'eye', 'fixation', 'motor', 'move', 'movement', 'moving', 'stand', 'standing', 'stood', 'crawl', 'crawled', 'crawling', 'walk', 'walking', 'walked', 'run', 'ran', 'running', 'jump', 'jumped', 'jumping', 'gait', 'locomotion', 'locomotory', 'body', 'posture', 'pose', 'gesture', 'face', 'facial', 'voice', 'speech', 'speak', 'talk', 'game', 'play', 'social', 'interaction', 'conversation', 'communication', 'emotion', 'emotional', 'stress', 'anxiety', 'rest', 'relax', 'sleep']
diagnosis_words = ['diagnosis', 'diagnostics', 'diagnostic', 'diagnose', 'diagnosing', 'diagnosed', 'screen', 'screened', 'screening', 'assessment', 'assess', 'assessing', 'assessed', 'detect', 'detection', 'detecting', 'detected']

In [ ]:
#for filter_words in [autism_words, diagnosis_words, ai_words, behavior_words]:
for filter_words in [autism_words, ai_words, behavior_words, diagnosis_words]:

  df = df[
      df['Title'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      df['Abstract'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      df['Keywords'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False)
  ]
  print(len(df))

114879
4928
3916
2229


In [ ]:
df.to_csv(f'{main_folder}/autism_ai_behavior_diagnosis_formatted__controlled_combined_noduplicates.csv', index=False)

# EXTRA WORK FOR AUTOMATING THE PROCESS!

In [ ]:
## force removing books' intros
df = df[~df['Abstract'].str.startswith("The proceedings contain ")]
df = df[~df['Title'].str.startswith("Table of contents")]
df = df[~df['Title'].str.contains("Proceedings")]
print(len(df))

2058


In [ ]:
#!pip install transformers

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
tqdm.pandas()

In [ ]:
def compute_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Flatten the tensor along the batch dimension
    return outputs.last_hidden_state.mean(dim=1).squeeze()

In [ ]:
df['text'] = (
    'Title: ' + df['Title'].astype(str) +
    '\n Abstract: ' + df['Abstract'].astype(str) +
    '\n Keywords: ' + df['Keywords'].astype(str) +
    '\n Authors: ' + df['Authors'].astype(str) +
    '\n Venue: ' + df['Venue'].astype(str) +
    '\n Year: ' + df['Year'].astype(str)
)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
grouped = df.groupby(['Title', 'Year'])

In [ ]:
def process_group(group):
    # If there's only one item in the group, no need to compute similarity
    if len(group) == 1:
        return []

    embeddings = group['text'].progress_apply(compute_embeddings)
    embeddings_2d = torch.stack(embeddings.to_list()).numpy()
    similarity_matrix = cosine_similarity(embeddings_2d)
    print(similarity_matrix)
    threshold = 0.95 ## SET A GOOD VALUE HERE!!!
    indices_to_drop = set()

    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            if similarity_matrix[i][j] > threshold:
                # Prefer rows with non-null DOI
                if group['DOI'].iloc[i] is not None and pd.notna(group['DOI'].iloc[i]):
                    indices_to_drop.add(group.index[j])
                else:
                    indices_to_drop.add(group.index[i])

    return list(indices_to_drop)

In [ ]:
# Apply the process_group function to each group
indices_to_drop = set()
for name, group in tqdm(grouped):
    indices_to_drop.update(process_group(group))

  0%|          | 3/1833 [00:02<23:13,  1.31it/s]

[[0.9999999 0.9842861]
 [0.9842861 1.0000004]]



  1%|          | 11/1833 [00:04<12:10,  2.49it/s]

[[1.         0.98793435]
 [0.98793435 0.99999964]]



  2%|▏         | 29/1833 [00:06<06:01,  4.99it/s]

...

[[1.         0.98040354]
 [0.98040354 1.        ]]



 98%|█████████▊| 1804/1833 [09:54<00:08,  3.53it/s]

[[1.0000001  0.9941555 ]
 [0.9941555  0.99999976]]



100%|██████████| 1833/1833 [09:56<00:00,  3.07it/s]

[[1.0000001  0.98411435]
 [0.98411435 1.        ]]


In [ ]:
indices_to_drop

{56,
 74,
 75,
 76,
 77,
 78,
 82,
 83,
 89,
 90,
 94,
 95,
 97,
 99,
 100,
 103,
 104,
 106,
 115,
 116,
 117,
 119,
 120,
 124,
 125,
 127,
 128,
 130,
 131,
 134,
 139,
 143,
 146,
 147,
 149,
 154,
 158,
 160,
 161,
 162,
 165,
 166,
 167,
 168,
 169,
 171,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 181,
 182,
 185,
 187,
 188,
 191,
 193,
 194,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 206,
 207,
 208,
 209,
 210,
 212,
 215,
 216,
 218,
 219,
 220,
 223,
 226,
 227,
 228,
 229,
 231,
 233,
 235,
 236,
 239,
 240,
 241,
 242,
 243,
 244,
 245,
 246,
 247,
 248,
 250,
 251,
 252,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 267,
 269,
 270,
 271,
 272,
 277,
 279,
 280,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 292,
 295,
 299,
 300,
 301,
 302,
 303,
 304,
 308,
 309,
 310,
 314,
 316,
 318,
 319,
 320,
 322,
 323,
 325,
 328,
 330,
 331,
 334,
 338,
 339,
 342,
 344,
 347,
 348,
 350,
 351,
 353,
 354,
 356,
 357,
 359,
 360,
 362,
 365,
 366,
 368,
 372,
 373,


In [ ]:
len(indices_to_drop)

221

In [ ]:
df[df['Title']=='Classification of Autism Spectrum Disorder From EEG-Based Functional Brain Connectivity Analysis.']

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database,text
233,Classification of Autism Spectrum Disorder Fro...,Autism is a psychiatric condition that is typi...,"AUTISM spectrum disorders, CHILDREN with autis...",NaN,https://search.ebscohost.com/login.aspx?direct...,"Alotaibi, Noura and Maharatna, Koushik",Neural Computation,2021.0,CINAHL (Exodus),Title: Classification of Autism Spectrum Disor...
240,Classification of Autism Spectrum Disorder Fro...,Autism is a psychiatric condition that is typi...,"AUTISM spectrum disorders, CHILDREN with autis...",NaN,https://search.ebscohost.com/login.aspx?direct...,"Alotaibi, Noura and Maharatna, Koushik",Neural Computation,2021.0,CINAHL (Exodus),Title: Classification of Autism Spectrum Disor...
899,Classification of Autism Spectrum Disorder Fro...,Autism is a psychiatric condition that is typi...,NaN,10.1162/neco_a_01394,https://pubmed.ncbi.nlm.nih.gov/34411269/,"Alotaibi, Maharatna",Neural computation,2021.0,PubMed,Title: Classification of Autism Spectrum Disor...


In [ ]:
# Drop identified duplicate indices
df = df.drop(list(indices_to_drop)).reset_index(drop=True)


In [ ]:
'''
df = df.sort_values(by='DOI', ascending=False)
df = df.drop_duplicates(subset=['Title', 'Year', 'Venue'])
df = df.reset_index(drop=True)
df
'''

"\ndf = df.sort_values(by='DOI', ascending=False)\ndf = df.drop_duplicates(subset=['Title', 'Year', 'Venue'])\ndf = df.reset_index(drop=True)\ndf\n"

In [ ]:
df['Title'].value_counts().head(50)

MOD-DHGN for Autism Segmentation                                                                                                                                   2
Ripplet II transform and higher order cumulants from R-fMRI data for diagnosis of autism                                                                           2
Mobile detection of autism through machine learning on home video: A development and prospective validation study.                                                 2
Hierarchical POMDP Framework for a Robot-Assisted ASD Diagnostic Protocol                                                                                          2
Sensing Technologies for Autism Spectrum Disorder Screening and Intervention.                                                                                      2
Miniature Crane Prototype: A Novel Approach for Recognizing Yoga Postures to Treat Autistic Children                                                               2
Autism Spe

In [ ]:
df[df['Title']=="Concordance between physiological arousal and emotion expression during fear in young children with autism spectrum disorders."]

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database,text
96,Concordance between physiological arousal and ...,This study aimed to measure emotional expressi...,"DIAGNOSIS of autism, TREATMENT of autism, AROU...",NaN,https://search.ebscohost.com/login.aspx?direct...,"Zantinge, Gemma and van Rijn, Sophie and Sto...",Autism: The International Journal of Research ...,2019.0,CINAHL (Exodus),Title: Concordance between physiological arous...
817,Concordance between physiological arousal and ...,This study aimed to measure emotional expressi...,"['arousal', 'autism', 'concordance', 'expressi...",10.1177/1362361318766439,https://pubmed.ncbi.nlm.nih.gov/29595334/,"Zantinge, van Rijn, Stockmann, Swaab",Autism : the international journal of research...,2019.0,PubMed,Title: Concordance between physiological arous...


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.sort_values('Title', inplace = True)
df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database,text
708,"""Mommy Blogs"" and the Vaccination Exemption Na...",Social media offer an unprecedented opportunit...,"['Internet', 'attitudes', 'health knowledge', ...",10.2196/publichealth.6586,https://pubmed.ncbi.nlm.nih.gov/27876690/,"Tangherlini, Roychowdhury, Glenn, Crespi, Band...",JMIR public health and surveillance,2016.0,PubMed,"Title: ""Mommy Blogs"" and the Vaccination Exemp..."
712,"""Sequencing Matters"": Investigating Suitable A...",Social robots have been shown to be promising ...,"['action selection', 'attention skills', 'auti...",10.3389/frobt.2022.784249,https://pubmed.ncbi.nlm.nih.gov/35356059/,"Baraka, Couto, Melo, Paiva, Veloso",Frontiers in robotics and AI,2022.0,PubMed,"Title: ""Sequencing Matters"": Investigating Sui..."
438,"""Um"" and ""Uh"" Usage Patterns in Children with ...","Pragmatic language difficulties, including unu...","['Autism', 'Disfluency', 'Fillers', 'Natural l...",10.1007/s10803-022-05565-4,https://pubmed.ncbi.nlm.nih.gov/35499654/,"Lawley, Bedrick, MacFarlane, Dolata, Salem, Fo...",Journal of autism and developmental disorders,2023.0,PubMed,"Title: ""Um"" and ""Uh"" Usage Patterns in Childre..."
1039,'SenseA'-Autism Early Signs and Pre-Aggressive...,This paper presents an efficient solution for ...,autism early signs; computer vision; feature e...,10.1109/AMS.2017.28,https://www.scopus.com/inward/record.uri?eid=2...,Gamaethige C.; Gunathilake U.; Jayasena D.; Ma...,AMS 2017 - Asia Modelling Symposium 2017 and 1...,2018.0,Scopus,Title: 'SenseA'-Autism Early Signs and Pre-Agg...
888,25th Annual Computational Neuroscience Meeting...,A1 Functional advantages of cell-type heteroge...,NaN,10.1186/s12868-016-0283-6,https://pubmed.ncbi.nlm.nih.gov/27534393/,NaN,BMC neuroscience,2016.0,PubMed,Title: 25th Annual Computational Neuroscience ...
...,...,...,...,...,...,...,...,...,...,...
1467,rs-fMRI Analysis Using Spatio-Temporal Sparse ...,Neuropsychiatric diseases such as Autism Spect...,CNN; Deep Learning; fMRI; Image Processing; Su...,10.1109/SIU55565.2022.9864751,https://www.scopus.com/inward/record.uri?eid=2...,Yener F.M.; Yildiz S.; Hafeez M.A.; Kayasandik...,2022 30th Signal Processing and Communications...,2022.0,Scopus,Title: rs-fMRI Analysis Using Spatio-Temporal ...
469,rs-fMRI and machine learning for ASD diagnosis...,Autism Spectrum Disorder (ASD) diagnosis is st...,NaN,10.1038/s41598-022-09821-6,https://pubmed.ncbi.nlm.nih.gov/35411059/,"Santana, de Carvalho, Rodrigues, Bastos, de So...",Scientific reports,2022.0,PubMed,Title: rs-fMRI and machine learning for ASD di...
1440,sha-Early Intervention for children at risk of...,Autism Spectrum Disorder (ASD) is a developmen...,Autism Spectrum Disorder; CNN; Cognitive devel...,10.1109/I4Tech55392.2022.9952803,https://www.scopus.com/inward/record.uri?eid=2...,Shetty T.; Zope V.; Dandekar M.; Devnani A.; M...,2022 International Conference on Industry 4.0 ...,2022.0,Scopus,Title: sha-Early Intervention for children at ...
1103,‘Autistic Robots’ for Embodied Emulation of Be...,The goal of this work is to enable interaction...,NaN,10.1007/978-3-319-70022-9_11,https://www.scopus.com/inward/record.uri?eid=2...,Baraka K.; Melo F.S.; Veloso M.,Lecture Notes in Computer Science (including s...,2017.0,Scopus,Title: ‘Autistic Robots’ for Embodied Emulatio...


In [ ]:
df.to_csv(f'{main_folder}/autism_ai_behavior_diagnosis_formatted__controlled_combined_noduplicates_semanticallyfiltered.csv', index=False)

In [ ]:
df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database,text
708,"""Mommy Blogs"" and the Vaccination Exemption Na...",Social media offer an unprecedented opportunit...,"['Internet', 'attitudes', 'health knowledge', ...",10.2196/publichealth.6586,https://pubmed.ncbi.nlm.nih.gov/27876690/,"Tangherlini, Roychowdhury, Glenn, Crespi, Band...",JMIR public health and surveillance,2016.0,PubMed,"Title: ""Mommy Blogs"" and the Vaccination Exemp..."
712,"""Sequencing Matters"": Investigating Suitable A...",Social robots have been shown to be promising ...,"['action selection', 'attention skills', 'auti...",10.3389/frobt.2022.784249,https://pubmed.ncbi.nlm.nih.gov/35356059/,"Baraka, Couto, Melo, Paiva, Veloso",Frontiers in robotics and AI,2022.0,PubMed,"Title: ""Sequencing Matters"": Investigating Sui..."
438,"""Um"" and ""Uh"" Usage Patterns in Children with ...","Pragmatic language difficulties, including unu...","['Autism', 'Disfluency', 'Fillers', 'Natural l...",10.1007/s10803-022-05565-4,https://pubmed.ncbi.nlm.nih.gov/35499654/,"Lawley, Bedrick, MacFarlane, Dolata, Salem, Fo...",Journal of autism and developmental disorders,2023.0,PubMed,"Title: ""Um"" and ""Uh"" Usage Patterns in Childre..."
1039,'SenseA'-Autism Early Signs and Pre-Aggressive...,This paper presents an efficient solution for ...,autism early signs; computer vision; feature e...,10.1109/AMS.2017.28,https://www.scopus.com/inward/record.uri?eid=2...,Gamaethige C.; Gunathilake U.; Jayasena D.; Ma...,AMS 2017 - Asia Modelling Symposium 2017 and 1...,2018.0,Scopus,Title: 'SenseA'-Autism Early Signs and Pre-Agg...
888,25th Annual Computational Neuroscience Meeting...,A1 Functional advantages of cell-type heteroge...,NaN,10.1186/s12868-016-0283-6,https://pubmed.ncbi.nlm.nih.gov/27534393/,NaN,BMC neuroscience,2016.0,PubMed,Title: 25th Annual Computational Neuroscience ...
...,...,...,...,...,...,...,...,...,...,...
1467,rs-fMRI Analysis Using Spatio-Temporal Sparse ...,Neuropsychiatric diseases such as Autism Spect...,CNN; Deep Learning; fMRI; Image Processing; Su...,10.1109/SIU55565.2022.9864751,https://www.scopus.com/inward/record.uri?eid=2...,Yener F.M.; Yildiz S.; Hafeez M.A.; Kayasandik...,2022 30th Signal Processing and Communications...,2022.0,Scopus,Title: rs-fMRI Analysis Using Spatio-Temporal ...
469,rs-fMRI and machine learning for ASD diagnosis...,Autism Spectrum Disorder (ASD) diagnosis is st...,NaN,10.1038/s41598-022-09821-6,https://pubmed.ncbi.nlm.nih.gov/35411059/,"Santana, de Carvalho, Rodrigues, Bastos, de So...",Scientific reports,2022.0,PubMed,Title: rs-fMRI and machine learning for ASD di...
1440,sha-Early Intervention for children at risk of...,Autism Spectrum Disorder (ASD) is a developmen...,Autism Spectrum Disorder; CNN; Cognitive devel...,10.1109/I4Tech55392.2022.9952803,https://www.scopus.com/inward/record.uri?eid=2...,Shetty T.; Zope V.; Dandekar M.; Devnani A.; M...,2022 International Conference on Industry 4.0 ...,2022.0,Scopus,Title: sha-Early Intervention for children at ...
1103,‘Autistic Robots’ for Embodied Emulation of Be...,The goal of this work is to enable interaction...,NaN,10.1007/978-3-319-70022-9_11,https://www.scopus.com/inward/record.uri?eid=2...,Baraka K.; Melo F.S.; Veloso M.,Lecture Notes in Computer Science (including s...,2017.0,Scopus,Title: ‘Autistic Robots’ for Embodied Emulatio...


In [ ]:
removed_rows = original_data[~original_data.index.isin(df2.index)]

# Check if there are removed rows
if not removed_rows.empty:
    # Randomly select one removed row and print it
    random_entry = removed_rows.sample()
    print(random_entry["Title"].values)
    print(random_entry["Abstract"].values)
    print(random_entry["Keywords"].values)
else:
    print("No entries were removed.")

['Exploratory Study of Parenting Differences for Autism Spectrum Disorder and Attachment Disorder.']
['The current study explored similarities and differences in parenting stress (PSI) and behaviours in parent reports of autism spectrum disorder (ASD) and attachment disorder (AD). 155 parents whose children had developmental delays and disorders completed the social communication questionnaire, Randolph attachment questionnaire, strengths and difficulties questionnaire, PSI, and parent-child relationship inventory. Parents of children with AD reported greater levels of PSI than parents of children with ASD. Parents of children reaching criteria for both disorders reported the greatest levels of PSI. Limit setting was poorest in parents of children with both classifications, followed by parents of children with AD, and then ASD. Limit setting mediated the relationship between PSI and child behaviour problems for parents of children with ASD\u2009<\u2009but not for parents of children wi

In [ ]:
for filter_words in [autism_sigle]:

  removed_rows = removed_rows[
      removed_rows['Title'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      removed_rows['Abstract'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      removed_rows['Keywords'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False)
  ]
  print(len(removed_rows))

4


In [ ]:
removed_rows.to_csv(f'{main_folder}/sciencedirect/unrelated_acronyms_formatted.csv', index=False)

In [ ]:
removed_rows.shape

In [ ]:
# Check if there are removed rows
if not df.empty:
    # Randomly select one removed row and print it
    random_entry = df.sample()
    print(random_entry["Title"].values)
    print(random_entry["Abstract"].values)
    print(random_entry["Keywords"].values)
else:
    print("No entries were removed.")